# 🎓 AegisX — Stage 2: SFT of YOUR OWN model (pure zero)

**No Qwen, no third-party base model.** This notebook takes the model you
pre-trained from scratch (`aegisx_train_colab.ipynb`) and fine-tunes it on
the 925 bilingual instruction rows, so it learns to *answer questions* in
addition to *speaking*.

Pipeline: ① pre-train from zero → ② **this step (SFT own model)** → ③ manual upload.

Requires a finished stage-1 checkpoint on Drive:
`MyDrive/aegisx/checkpoints/aegisx-mini/model.pt` + `tokenizer.json`.

## 1. Setup

In [ ]:
!pip install -q torch

import os
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('Skipping Drive mount.')

## 2. Get the AegisX code + instruction data

In [ ]:
WORK = '/content/aegisx'
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
if not os.path.isdir('.git'):
    !git clone https://github.com/FerzDevZ/AegisX.git .
else:
    !git -C . pull --ff-only
print('Repo ready.')

In [ ]:
# Build the SFT chat text from data/finetune/instructions.jsonl (925 rows).
!python scripts/build_sft_text.py --out data/finetune/sft_chat.txt

# Small dataset dir just for this SFT step, so stage-1 raw corpus is untouched.
SFT_DATA = '/content/aegisx-sft-data'
os.makedirs(SFT_DATA, exist_ok=True)
!cp data/finetune/sft_chat.txt {SFT_DATA}/sft.txt
print('SFT data ready at', SFT_DATA)

## 3. Point at your stage-1 checkpoint

In [ ]:
# Your model from stage 1 (pre-trained from scratch).
INIT_MODEL = '/content/drive/MyDrive/aegisx/checkpoints/aegisx-mini/model.pt'
assert os.path.exists(INIT_MODEL), f'Not found: {INIT_MODEL} - run stage 1 first'

# New output dir (keep stage-1 checkpoint untouched).
SFT_OUT = '/content/drive/MyDrive/aegisx/checkpoints/aegisx-sft' if USE_DRIVE else '/content/aegisx/checkpoints/aegisx-sft'

# Kalau SFT pernah putus di tengah, lanjut dari checkpoint SFT terakhir.
RESUME_FROM = ''
for cand in [f'{SFT_OUT}/model_latest.pt', f'{SFT_OUT}/model.pt']:
    if os.path.exists(cand):
        RESUME_FROM = cand
        print('⏳ SFT checkpoint ditemukan - RESUME dari', cand)
        break
print('init from:', RESUME_FROM or INIT_MODEL)
print('sft out :', SFT_OUT)

## 4. SFT train (your own model)

`--init-from` loads your weights and continues training on the instruction
text with a gentler LR (auto 1e-4) so it learns to answer without forgetting
pre-training. Fewer steps + early stopping keeps it fast and safe.

In [ ]:
# --- SFT hyperparameters ---
MAX_STEPS    = 1500
BATCH_SIZE   = 16
GRAD_ACCUM   = 4
EVAL_EVERY   = 250
EARLY_STOP   = 4      # stop after 4 evals without val improvement
LR           = 1e-4   # gentle: don't destroy stage-1 knowledge
WARMUP       = 100
DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)

In [ ]:
# Architecture args are ignored when --init-from is set (checkpoint wins).
!python -m aegisx.train \
    --data {SFT_DATA} \
    --out {SFT_OUT} \
    --init-from {RESUME_FROM if RESUME_FROM else INIT_MODEL} \
    --max-steps {MAX_STEPS} \
    --batch-size {BATCH_SIZE} \
    --grad-accum {GRAD_ACCUM} \
    --eval-every {EVAL_EVERY} \
    --early-stop-patience {EARLY_STOP} \
    --lr {LR} \
    --warmup-steps {WARMUP} \
    --device {DEVICE}

## 5. Compare: before vs after SFT

In [ ]:
import os
if os.path.exists(f'{SFT_OUT}/model.pt'):
    # 1. eval the ORIGINAL stage-1 model (baseline)
    if os.path.exists(INIT_MODEL):
        !python -m aegisx.eval --model {INIT_MODEL} \
            --tokenizer {SFT_OUT}/tokenizer.json --device {DEVICE} --max-new-tokens 80 2>/dev/null | tail -4
        print('--- after SFT ---')
    # 2. eval the SFT model
    !python -m aegisx.eval --model {SFT_OUT}/model.pt \
        --tokenizer {SFT_OUT}/tokenizer.json --device {DEVICE} --max-new-tokens 80 2>/dev/null | tail -4
else:
    print('Skipped: SFT model not found - check the training cell.')

## 6. Chat with the SFT model

In [ ]:
import os
if os.path.exists(f'{SFT_OUT}/model.pt'):
    !python -m aegisx.chat --model {SFT_OUT}/model.pt --tokenizer {SFT_OUT}/tokenizer.json \
        --prompt "You are AegisX, a cybersecurity assistant. User: Apa itu SQL injection?\n\nAegisX:" \
        --max-new-tokens 150 --temperature 0.7 --top-k 50
else:
    print('Skipped: SFT model not found.')

## 7. Export for manual Hugging Face upload

Same packaging as stage 1: model.pt + tokenizer.json + config + model card
+ knowledge/ folder, zipped in your Drive. Upload it manually.

In [ ]:
import shutil, zipfile
from pathlib import Path

if os.path.exists(f'{SFT_OUT}/model.pt'):
    EXPORT_DIR = Path('/content/drive/MyDrive/aegisx/export/aegisx-mini-sft') if USE_DRIVE else Path('/content/aegisx/export/aegisx-mini-sft')
    if EXPORT_DIR.exists():
        shutil.rmtree(EXPORT_DIR)
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy(f'{SFT_OUT}/model.pt', EXPORT_DIR / 'model.pt')
    shutil.copy(f'{SFT_OUT}/tokenizer.json', EXPORT_DIR / 'tokenizer.json')
    shutil.copy(f'{SFT_OUT}/config.json', EXPORT_DIR / 'config.json')

    card = Path('hf/MODEL_CARD.md')
    if card.exists():
        shutil.copy(card, EXPORT_DIR / 'README.md')

    KNOW = EXPORT_DIR / 'knowledge'
    KNOW.mkdir(exist_ok=True)
    for f in sorted(Path('data/raw').glob('*.txt')):
        shutil.copy(f, KNOW / f.name)

    zip_path = Path(str(EXPORT_DIR) + '.zip')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for f in sorted(EXPORT_DIR.iterdir()):
            if f.is_dir():
                for inner in f.rglob('*'):
                    if inner.is_file():
                        zf.write(inner, arcname=f'{f.name}/{inner.name}')
            else:
                zf.write(f, arcname=f.name)
    print('Export folder:')
    for f in sorted(EXPORT_DIR.iterdir()):
        print(f'  {f.name}')
    print(f'ZIP: {zip_path}')
else:
    print('Skipped: SFT model not found.')